# AWGN Eksik Metrik Tamamlama: F1 Score & MCC

Daha önce Drive'a **yalnızca `acc.pkl`** kaydedilmiş olan AWGN sonuçları için
eksik **`f1_macro_scores.pkl`** ve **`mcc_metric_scores.pkl`** dosyalarını üretir ve Drive'a kaydeder.

**Eğitim yapılmaz** — mevcut ağırlıklar yüklenerek sadece inference + metrik hesabı yapılır.

Bu notebook çalıştırıldıktan sonra **09_results_analysis_and_comparison.ipynb** tekrar çalıştırılırsa
AWGN satırı dahil tüm üç-yönlü (AWGN Baseline / Before FT / After FT) grafikleri eksiksiz üretilir.

In [ ]:
import tensorflow as tf
import numpy as np
import pickle
import os
from sklearn.metrics import f1_score, matthews_corrcoef

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "https://github.com/erigami-sl/AMR-UnderDifferentNoises-DL.git"
PROJECT_DIR = "/content/AMR-UnderDifferentNoises-DL"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL}
else:
    print(f"Proje mevcut: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, PROJECT_DIR)
print(f"Çalışma dizini: {os.getcwd()}")

In [ ]:
DRIVE_BASE    = '/content/drive/MyDrive/AMR-Project'
DATASET_PATH  = os.path.join(DRIVE_BASE, 'RML2016.10a_dict.pkl')
RESULTS_BASE  = os.path.join(DRIVE_BASE, 'results')

MCLDNN_DIR    = os.path.join(RESULTS_BASE, 'mcldnn_awgn')
PETCGDNN_DIR  = os.path.join(RESULTS_BASE, 'petcgdnn_awgn')

BATCH_SIZE = 400

for d in [MCLDNN_DIR, PETCGDNN_DIR]:
    if not os.path.isdir(d):
        raise FileNotFoundError(f"Klasör bulunamadı: {d}")

print(f"Dataset : {DATASET_PATH}")
print(f"MCLDNN  : {MCLDNN_DIR}")
print(f"PETCGDNN: {PETCGDNN_DIR}")
print("Klasörler OK.")

In [ ]:
from src.utils.dataset import load_data, prepare_data_mcldnn, prepare_data_petcgdnn

(mods, snrs, lbl), \
(X_train, Y_train), (X_val, Y_val), \
(X_test, Y_test), (train_idx, val_idx, test_idx) = load_data(DATASET_PATH)

print(f"Modülasyonlar ({len(mods)}): {mods}")
print(f"SNR: {snrs[0]} → {snrs[-1]} dB  ({len(snrs)} seviye)")
print(f"Test: X={X_test.shape}, Y={Y_test.shape}")

In [ ]:
def compute_and_save_f1_mcc(model, X_inputs, Y_test, lbl, test_idx, snrs, save_dir):
    test_SNRs  = [lbl[x][1] for x in test_idx]
    f1_scores  = {}
    mcc_scores = {}

    for snr in snrs:
        mask  = np.where(np.array(test_SNRs) == snr)
        X_i   = [inp[mask] for inp in X_inputs]
        Y_i   = Y_test[mask]
        Y_hat = model.predict(X_i, batch_size=BATCH_SIZE, verbose=0)

        y_true = np.argmax(Y_i,   axis=1)
        y_pred = np.argmax(Y_hat, axis=1)

        f1_scores[snr]  = f1_score(y_true, y_pred, average='macro')
        mcc_scores[snr] = matthews_corrcoef(y_true, y_pred)
        print(f"  SNR {snr:+3d} dB  |  F1={f1_scores[snr]:.4f}  MCC={mcc_scores[snr]:.4f}")

    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, 'f1_macro_scores.pkl'),   'wb') as f: pickle.dump(f1_scores,  f)
    with open(os.path.join(save_dir, 'mcc_metric_scores.pkl'), 'wb') as f: pickle.dump(mcc_scores, f)

    print(f"\n  Ort. F1={np.mean(list(f1_scores.values())):.4f}  "
          f"Ort. MCC={np.mean(list(mcc_scores.values())):.4f}")
    print(f"  Kaydedildi → {save_dir}")
    return f1_scores, mcc_scores


def load_weights_from_drive(model, drive_dir):
    candidates = [
        'best_weights_finetuned.keras',
        'best_weights.keras',
        'best_weights.h5',
        'best_weights.weights.h5',
        os.path.join('weights', 'best_weights_finetuned.keras'),
        os.path.join('weights', 'best_weights.keras'),
        os.path.join('weights', 'best_weights.h5'),
    ]
    for fname in candidates:
        path = os.path.join(drive_dir, fname)
        if os.path.exists(path):
            model.load_weights(path)
            print(f"Ağırlıklar yüklendi: {path}")
            return path

    # Bulunamazsa klasör içeriğini listele
    print(f"HATA: Ağırlık dosyası bulunamadı: {drive_dir}")
    print(f"Klasör içeriği: {os.listdir(drive_dir) if os.path.isdir(drive_dir) else 'klasör yok'}")
    raise FileNotFoundError(f"Ağırlık dosyası bulunamadı: {drive_dir}")


print("Yardımcı fonksiyonlar tanımlandı.")

## 1. MCLDNN — AWGN F1 & MCC

In [ ]:
from src.models.mcldnn import MCLDNN

mcldnn = MCLDNN(weights=None, classes=len(mods))
mcldnn.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
load_weights_from_drive(mcldnn, MCLDNN_DIR)

data_mcldnn = prepare_data_mcldnn(X_train, X_val, X_test)
print(f"MCLDNN test input: {data_mcldnn['test'][0].shape}")

In [ ]:
print("=== MCLDNN AWGN — F1 & MCC ===")
mcldnn_awgn_f1, mcldnn_awgn_mcc = compute_and_save_f1_mcc(
    model=mcldnn, X_inputs=data_mcldnn['test'],
    Y_test=Y_test, lbl=lbl, test_idx=test_idx,
    snrs=snrs, save_dir=MCLDNN_DIR
)

## 2. PET-CGDNN — AWGN F1 & MCC

In [ ]:
from src.models.petcgdnn import PETCGDNN, _cos, _sin

petcgdnn = PETCGDNN(weights=None, classes=len(mods))
petcgdnn.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

try:
    load_weights_from_drive(petcgdnn, PETCGDNN_DIR)
except Exception as e:
    print(f"load_weights başarısız ({e}), tam model yükleniyor...")
    for fname in ['best_weights_finetuned.keras', 'best_weights.keras']:
        path = os.path.join(PETCGDNN_DIR, fname)
        if os.path.exists(path):
            petcgdnn = tf.keras.models.load_model(
                path, custom_objects={'_cos': _cos, '_sin': _sin}, safe_mode=False)
            print(f"Tam model yüklendi: {path}")
            break

data_petcgdnn = prepare_data_petcgdnn(X_train, X_val, X_test)
print(f"PET-CGDNN test input: {data_petcgdnn['test'][0].shape}")

In [ ]:
print("=== PET-CGDNN AWGN — F1 & MCC ===")
petcgdnn_awgn_f1, petcgdnn_awgn_mcc = compute_and_save_f1_mcc(
    model=petcgdnn, X_inputs=data_petcgdnn['test'],
    Y_test=Y_test, lbl=lbl, test_idx=test_idx,
    snrs=snrs, save_dir=PETCGDNN_DIR
)

## 3. Doğrulama — AWGN Dosyaları Var Mı?

In [ ]:
expected = [
    (MCLDNN_DIR,   'acc.pkl'),
    (MCLDNN_DIR,   'f1_macro_scores.pkl'),
    (MCLDNN_DIR,   'mcc_metric_scores.pkl'),
    (PETCGDNN_DIR, 'acc.pkl'),
    (PETCGDNN_DIR, 'f1_macro_scores.pkl'),
    (PETCGDNN_DIR, 'mcc_metric_scores.pkl'),
]

print(f"{'Model':<12} {'Metrik':<25} {'Durum'}")
print('-' * 50)
for folder, fname in expected:
    model_tag = 'mcldnn' if 'mcldnn' in folder else 'petcgdnn'
    ok = os.path.exists(os.path.join(folder, fname)) or \
         os.path.exists(os.path.join(folder, 'predictions', fname))
    print(f"{model_tag:<12} {fname:<25} {'✅' if ok else '❌  EKSIK'}")

print('\n09_results_analysis_and_comparison.ipynb artık tam veri ile çalışabilir.')